<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300" height="100"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<center><font size=10>AI Agents for Business Applications</center></font>
<center><font size=6>Prompt Engineering and Retrieval Augmented Generation - Week 1</center></font>

<center><p float="center">
  <img src="https://i.ibb.co/Q325rK84/medical.png" width="480"/>
</p></center>

<center><font size=6>LLM-Powered Medical Assistant</center></font>

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

1. **Critical Care Protocols:** "What is the protocol for managing sepsis in a critical care unit?"

2. **General Surgery:** "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

3. **Dermatology:** "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

4. **Neurology:** "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Install required libraries
!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0 \
              datasets==4.0.0 \
              evaluate==0.4.5 \
              langchain_openai==0.3.30

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.8 requires langchain-core>=1.0.0, but you have langchain-core 0.3.83 which is incompatible.


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [2]:
# Import core libraries
import os                                                                       # Interact with the operating system (e.g., set environment variables)
import json                                                                     # Read/write JSON data

# Import libraries for working with PDFs and OpenAI
from langchain.document_loaders import PyMuPDFLoader                            # Load and extract text from PDF files
from openai import OpenAI                                                       # Access OpenAI's models and services

# Import libraries for processing dataframes and text
import tiktoken                                                                 # Tokenizer used for counting and splitting text for models
import pandas as pd                                                             # Load, manipulate, and analyze tabular data

# Import LangChain components for data loading, chunking, embedding, and vector DBs
from langchain.text_splitter import RecursiveCharacterTextSplitter              # Break text into overlapping chunks for processing
from langchain.embeddings.openai import OpenAIEmbeddings                        # Create vector embeddings using OpenAI's models  # type: ignore
from langchain.vectorstores import Chroma                                       # Store and search vector embeddings using Chroma DB  # type: ignore


from datasets import Dataset                                                    # Used to structure the input (questions, answers, contexts etc.) in tabular format
from langchain_openai import ChatOpenAI                                         # This is needed since LLM is used in metric computation

## Question Answering using LLM

### OpenAI API Calling



In [3]:
# Load the JSON file and extract values
file_name = 'config.json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    API_KEY = config.get("OPENAI_API_KEY")                                             # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract the OpenAI base URL from the config

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = API_KEY                                          # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable

# Initialize OpenAI client
client = OpenAI()                                                               # Create an instance of the OpenAI client

### Defining the function to Generate a Response From the LLM

In [4]:
# Define a function to get a response
def response(user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",                                                     # Specify the model to use
        messages=[
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply                                                        # Execute the function with the prompts

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [5]:
question_1 = "What is the protocol for managing sepsis in a critical care unit?"
base_prompt_response_1 = response(question_1)
base_prompt_response_1

"Managing sepsis in a critical care unit involves a systematic approach to ensure timely identification, resuscitation, and treatment of the condition. Here’s an overview of the protocol typically followed:\n\n### 1. **Early Recognition and Diagnosis**\n   - **Screening:** Use tools like the Sequential Organ Failure Assessment (SOFA) score or the Quick SOFA (qSOFA) to identify patients at risk for sepsis.\n   - **Clinical Signs:** Monitor for signs of infection, organ dysfunction, and systemic inflammatory response syndrome (SIRS).\n\n### 2. **Initial Resuscitation (Within 1 hour of recognition)**\n   - **Fluid Resuscitation:** Administer intravenous fluids (typically crystalloids) promptly, generally 30 mL/kg within the first 3 hours.\n   - **Vasopressors:** If hypotension persists after adequate fluid resuscitation, initiate vasopressors (e.g., norepinephrine) to maintain mean arterial pressure (MAP) ≥ 65 mmHg.\n\n### 3. **Source Control**\n   - **Identify and Manage the Source of In

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [6]:
question_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
base_prompt_response_2 = response(question_2)
base_prompt_response_2

"Common symptoms of appendicitis include:\n\n1. **Abdominal Pain**: Typically starts near the belly button and shifts to the lower right abdomen. The pain usually becomes sharper over time.\n2. **Nausea and Vomiting**: Often accompanies the abdominal pain.\n3. **Loss of Appetite**: Many people with appendicitis experience a decrease in appetite.\n4. **Fever**: A low-grade fever may be present, which can increase if the appendix becomes more inflamed.\n5. **Constipation or Diarrhea**: Some individuals may experience changes in bowel habits.\n6. **Bloating or Gas**: Some may feel bloated or have difficulty passing gas.\n\nAppendicitis cannot be treated effectively with medication alone. The standard treatment for appendicitis is surgical removal of the appendix, a procedure known as an **appendectomy**. This can be performed using two main methods:\n\n1. **Open Appendectomy**: A larger incision is made in the lower right abdomen to remove the appendix.\n2. **Laparoscopic Appendectomy**: 

### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [7]:
question_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
base_prompt_response_3 = response(question_3)
base_prompt_response_3

'Sudden patchy hair loss, often referred to as alopecia areata, can manifest as localized bald spots on the scalp and other areas of the body. Here are some effective treatments and potential causes for this condition:\n\n### Possible Causes:\n1. **Autoimmune Disorders**: The body’s immune system may mistakenly attack hair follicles, leading to hair loss.\n2. **Genetic Factors**: A family history of alopecia or other autoimmune diseases may increase the risk.\n3. **Stress**: Physical or emotional stress can trigger hair loss in some individuals.\n4. **Hormonal Changes**: Hormonal shifts, such as those occurring during pregnancy or menopause, can lead to hair loss.\n5. **Nutritional Deficiencies**: Lack of essential nutrients (e.g., vitamins D, B12, iron, and zinc) can contribute to hair loss.\n6. **Infections**: Fungal infections like tinea capitis can cause patchy hair loss.\n7. **Other Medical Conditions**: Conditions like thyroid disorders, vitiligo, or skin diseases may be associat

### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [8]:
question_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
base_prompt_response_4 = response(question_4)
base_prompt_response_4

"Treatments for a person who has sustained a physical injury to brain tissue, such as a traumatic brain injury (TBI), can vary widely based on the severity of the injury, the specific areas of the brain affected, and the resulting impairments. Here are some common approaches used in the management and rehabilitation of brain injuries:\n\n### Immediate Medical Treatment\n1. **Emergency Care**: In cases of severe injury, immediate medical attention may involve life-saving measures such as maintaining airway, breathing, and circulation (ABCs), as well as monitoring intracranial pressure (ICP).\n2. **Medications**: \n   - **Diuretics** to reduce swelling in the brain.\n   - **Anticonvulsants** to prevent seizures.\n   - **Pain management** medications.\n   - **Sedatives** to help with agitation or anxiety.\n\n### Surgical Interventions\n- **Craniotomy**: Surgery to remove a portion of the skull to relieve pressure from swelling or to repair damaged brain tissue.\n- **Hemorrhage Control**: 

**Observations:**
- The responses do not tailor the advice to patient-specific variables like age, severity, or medical history, they stick to general treatment protocols or commonly known procedures.

- The answers provide basic overviews (e.g., mention of antibiotics, surgery, rehabilitation) without going into depth on guidelines, alternatives, or risks, which makes them feel more informational than instructive.


## Question Answering using LLM with Prompt Engineering

### Define a system prompt that aligns with the business problem

In [9]:
system_prompt = """
You are an AI assistant specializing in medical knowledge. Your role is to provide clear, precise, and medically reliable responses based on established medical guidelines and best practices.

When answering, prioritize factual correctness, align with widely accepted medical standards, and ensure clarity for both medical professionals and general users.
If a query requires specific reference materials beyond general medical knowledge, acknowledge the limitation rather than speculating.

"""

### Defining the function to Generate a Response From the LLM

In [10]:
# Define a function to get a response from the OpenAI chat model
def response(system_prompt, user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",                                                    # Specify the model to use (GPT-4o in this case)
        messages=[
            {"role": "system", "content": system_prompt},                       # System prompt sets the assistant's behavior
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output (0 = deterministic)
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply

### Question1: What is the protocol for managing sepsis in a critical care unit?

In [11]:
response_with_prompt_eng_1 = response(system_prompt, question_1)
response_with_prompt_eng_1

"The management of sepsis in a critical care unit follows established guidelines, such as those from the Surviving Sepsis Campaign. The protocol typically includes the following key components:\n\n### 1. **Early Recognition**\n   - Identify patients with signs of sepsis, which include fever, hypothermia, tachycardia, tachypnea, altered mental status, and signs of organ dysfunction.\n\n### 2. **Initial Resuscitation**\n   - **Fluid Resuscitation**: Administer intravenous (IV) fluids (usually crystalloids) within the first hour. The recommended initial fluid bolus is 30 mL/kg for adults.\n   - **Vasopressors**: If the patient remains hypotensive after fluid resuscitation (i.e., systolic blood pressure < 90 mmHg or mean arterial pressure < 65 mmHg), initiate vasopressors (e.g., norepinephrine) to maintain adequate perfusion.\n\n### 3. **Antibiotic Therapy**\n   - Administer broad-spectrum antibiotics within the first hour of recognition. Choose empiric therapy based on local microbiology 

### Question2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [12]:
response_with_prompt_eng_2 = response(system_prompt, question_2)
response_with_prompt_eng_2

'Common symptoms of appendicitis include:\n\n1. **Abdominal Pain**: Typically starts around the navel and then shifts to the lower right abdomen.\n2. **Nausea and Vomiting**: Often follows the onset of abdominal pain.\n3. **Loss of Appetite**: Patients may not feel like eating.\n4. **Fever**: Mild fever may develop.\n5. **Constipation or Diarrhea**: Changes in bowel habits can occur.\n6. **Abdominal Swelling**: In some cases, the abdomen may become distended.\n\nAppendicitis cannot be effectively treated with medication alone. The definitive treatment is surgical removal of the appendix, a procedure known as **appendectomy**. There are two common approaches to performing an appendectomy:\n\n1. **Open Appendectomy**: A larger incision is made in the lower right abdomen to remove the appendix.\n2. **Laparoscopic Appendectomy**: This is a minimally invasive technique using small incisions and a camera to guide the surgery.\n\nWhile antibiotics may be used to manage mild cases or as a temp

### Question3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [13]:
response_with_prompt_eng_3 = response(system_prompt, question_3)
response_with_prompt_eng_3

"Sudden patchy hair loss, often characterized by localized bald spots on the scalp, is commonly associated with a condition known as alopecia areata. Here are the potential causes, effective treatments, and solutions for this condition:\n\n### Possible Causes:\n1. **Alopecia Areata**: An autoimmune disorder where the immune system attacks hair follicles, leading to hair loss.\n2. **Stress or Trauma**: Psychological stress or physical trauma can trigger hair loss.\n3. **Genetic Predisposition**: Family history may increase susceptibility.\n4. **Hormonal Changes**: Factors like pregnancy, menopause, or thyroid dysfunction can play a role.\n5. **Nutritional Deficiencies**: Lack of essential nutrients such as iron, vitamin D, or B vitamins can contribute.\n6. **Medical Conditions**: Conditions such as vitiligo, psoriasis, or fungal infections (like tinea capitis) can cause hair loss.\n\n### Effective Treatments:\n1. **Topical Corticosteroids**: These anti-inflammatory medications can help 

### Question4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [14]:
response_with_prompt_eng_4 = response(system_prompt, question_4)
response_with_prompt_eng_4

"The treatment for a person who has sustained a physical injury to brain tissue, such as a traumatic brain injury (TBI), can vary significantly depending on the severity of the injury, the specific areas of the brain affected, and the resulting impairments. Here are common approaches to managing such injuries:\n\n1. **Initial Assessment and Stabilization**:\n   - Immediate medical evaluation is crucial. This may involve imaging studies like CT scans or MRIs to assess the extent of the injury.\n   - Stabilization of vital signs and prevention of secondary brain injury (e.g., controlling intracranial pressure).\n\n2. **Medical Management**:\n   - **Medications**: Depending on symptoms, medications may be prescribed for pain management, seizures, or to manage other symptoms such as anxiety or depression.\n   - **Neuroprotective agents**: Research is ongoing into drugs that may protect brain tissue immediately following injury.\n\n3. **Rehabilitation**:\n   - **Physical Therapy**: Helps im

**Observations:**

**Question1: Sepsis Management in Critical Care**

* **Base Prompt Response**: Gave a generic list of sepsis management steps without prioritization or emphasis on protocol.
* **Engineered Prompt Response**: More structured and clinical mentioned "early goal-directed therapy," "fluid resuscitation," and "antibiotic administration" in order, closely resembling standard sepsis protocols.

* Improved in clinical depth and sequencing.


**Question2: Appendicitis Symptoms and Treatment**

* **Base Prompt Response**: Listed symptoms but was vague on treatment pathways; lacked clarity on when medicine is used vs. surgery.
* **Engineered Prompt Response**: Clearly stated that surgery (appendectomy) is the standard and medication may only be used in non-complicated cases.

* Improved in decision-making clarity and completeness.

**Question3: Patchy Hair Loss (Alopecia Areata)**

* **Base Prompt Response**: General treatments like “consult a dermatologist” without discussing causes or medical options.
* **Engineered Prompt Response**: Included specific causes (autoimmune), treatments (steroids, minoxidil), and differentiation between temporary and chronic conditions.

* Improved in specificity and cause-treatment mapping.

**Question4: Brain Injury Treatment**

* **Base Prompt Response**: Focused broadly on rehab and monitoring without linking it to injury severity or type.
* **Engineered Prompt Response**: Mentioned both acute interventions (e.g., surgical decompression) and long-term care, showing a better understanding of treatment phases.
* Improved in handling both acute and chronic dimensions of treatment

## Data Preparation for RAG

### Loading the Data

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [15]:
# Set the path to the PDF file
manual_pdf_path = "medical_diagnosis_manual.pdf"                       # Path to the medical diagnosis manual PDF

# Load the PDF using LangChain's PyPDFLoader
pdf_loader = PyMuPDFLoader(manual_pdf_path)                                     # Initialize the PDF loader with the file path

# Extract content from the PDF
manual = pdf_loader.load()                                                      # Load and extract text from all pages of the PDF

### Data Overview

#### Checking the first 5 pages

In [16]:
# Loop through the first 5 pages of the PDF content
for i in range(5):
    print(f"Page Number : {i+1}", end="\n")                                     # Print the page number (1-based index)
    print(manual[i].page_content, end="\n")                                     # Print the content of the corresponding page

Page Number : 1
bhaskarjitsarmah@gmail.com
BHASKARJIT
This file is meant for personal use by bhaskarjitsarmah@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
bhaskarjitsarmah@gmail.com
BHASKARJIT
This file is meant for personal use by bhaskarjitsarmah@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ...........................................................................................................

### Data Chunking

#### Chunk the PDF into Manageable Text Sections Using a Token-Based Splitter

In [17]:
# Initialize a text splitter that uses OpenAI's token encoder
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',                                                # Encoding used by popular LLMs
    chunk_size=256,                                                             # Each chunk will have up to 256 tokens
    chunk_overlap=20                                                            # 20 tokens will overlap between consecutive chunks (for context continuity)
)

#### Split the Loaded PDF into Chunks for Further Processing

In [18]:
# Use the text splitter to divide the PDF content into smaller chunks
document_chunks = pdf_loader.load_and_split(text_splitter)                      # Returns a list of chunked documents

#### Check the Number of Chunks Created

In [19]:
len(document_chunks)                                                            # Total number of text chunks generated from the PDF

15671

### Generate Vector Embeddings for Text Chunks Using OpenAI

In [20]:
# Initialize the OpenAI Embeddings model with API credentials
embedding_model = OpenAIEmbeddings(
    openai_api_key=API_KEY,                                                     # Your OpenAI API key for authentication
    openai_api_base=OPENAI_API_BASE                                             # The OpenAI API base URL endpoint
)

# Generate embeddings (vector representations) for the first two document chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)      # Embedding for chunk 0
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)      # Embedding for chunk 1

# Check and print the dimension (length) of the embedding vector
print("Dimension of the embedding vector ", len(embedding_1))                   # Typically 1536 or 2048 depending on model

/var/folders/7f/3kysfv4x46z99821gn517yw40000gn/T/ipykernel_7474/470632182.py:2: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedding_model = OpenAIEmbeddings(


Dimension of the embedding vector  1536


In [21]:
# Verify if both embeddings have the same dimension (should be True)
len(embedding_1) == len(embedding_2)

# Return/display the two embedding vectors for further inspection or use
embedding_1, embedding_2

([-0.01637847837788081,
  0.0029138060957110954,
  0.013102782888569173,
  -0.010465915126126183,
  -0.007638759056074811,
  0.020986199005370937,
  -0.019341555865067278,
  -0.02040174003162086,
  -0.02450655202107084,
  -0.0005466571698408648,
  0.030201642122996475,
  -0.013007638036273,
  0.0067994469013274396,
  0.021978422696187626,
  -0.008141666297130997,
  0.0034184127677558923,
  0.021720172515858394,
  -0.007040706962722451,
  0.02363665942175489,
  0.006864009446409748,
  -0.00393491219709172,
  0.01972213341160669,
  -0.0058072241418333885,
  0.03822097825994938,
  -0.021652212040121504,
  -0.016595951527709794,
  0.022467737748964173,
  -0.03860155766913406,
  0.0006388285999836195,
  -0.02623274922237499,
  0.006860611515755168,
  -0.01148532319350216,
  -0.015454215162801004,
  -0.017506623020171277,
  -0.01631051790214392,
  0.0020761929062910146,
  0.02065998834930481,
  0.004060639356601757,
  0.02789098594794225,
  -0.009514468465809738,
  0.019395924618185845,
  -0

### Vector Database Creation

#### Setup Vector Store Directory

In [22]:
# Creating a folder for saving the vector DB so it persists between runs
out_dir = 'medical_db'                                                          # Directory to store the persistent vector database

# Create the directory if it doesn't exist
if not os.path.exists(out_dir):
    os.makedirs(out_dir)                                                        # Make directory to save vector store files

#### Create Vector Store from Documents

In [23]:
# Building the vector store and saving it to disk for future use
vectorstore = Chroma.from_documents(
    document_chunks,                                                            # Documents to index
    embedding_model,                                                            # Embedding model for converting text to vectors
    persist_directory=out_dir                                                   # Save vector DB files here
)

#### Load Vector Store

In [ ]:
# Reloading the vector store from disk without recomputing embeddings
vectorstore = Chroma(
    persist_directory=out_dir,                                                  # Load existing vector DB files
    embedding_function=embedding_model                                          # Use the same embedding function for queries
)

/tmp/ipython-input-4264619131.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


#### Explore Vector Store and Perform Searches

In [ ]:
# Inspect the embedding function in use
vectorstore.embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7d5ec6ccfce0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7d5ec41e8290>, model='text-embedding-ada-002', deployment='text-embedding-ada-002', openai_api_version='', openai_api_base='https://aibe.mygreatlearning.com/openai/v1', openai_api_type='', openai_proxy='', embedding_ctx_length=8191, openai_api_key='gl-U2FsdGVkX1/cNSMrkBY1fM7BOCCiGdAegq/qj0J6BZ2bzGsxaW7NfCOrXuFu99GD', openai_organization=None, allowed_special=set(), disallowed_special='all', chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None)

In [ ]:
# Search for top 3 most relevant chunk for the query
vectorstore.similarity_search(
    "What are the common symptoms and treatments for pulmonary embolism?",
    k=3
)

[Document(metadata={'source': '/content/medical_diagnosis_manual.pdf', 'total_pages': 4114, 'subject': '', 'modDate': "D:20140421075319+10'00'", 'format': 'PDF 1.6', 'author': '', 'keywords': '', 'producer': 'Atop CHM to PDF Converter', 'creationDate': 'D:20120615054440Z', 'moddate': '2014-04-21T07:53:19+10:00', 'file_path': '/content/medical_diagnosis_manual.pdf', 'trapped': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'page': 2079, 'creator': 'Atop CHM to PDF Converter', 'creationdate': '2012-06-15T05:44:40+00:00'}, page_content='Chapter 194. Pulmonary Embolism\nIntroduction\nPulmonary embolism (PE) is the occlusion of ≥ 1 pulmonary arteries by thrombi that originate\nelsewhere, typically in the large veins of the lower extremities or pelvis. Risk factors are\nconditions that impair venous return, conditions that cause endothelial injury or dysfunction,\nand underlying hypercoagulable states. Symptoms are nonspecific and include dyspnea,\npleuritic chest pain

### Retrieval and Response Generation using Vector Search

#### Convert Vector Store into a Retriever and Retrieve Relevant Documents

In [ ]:
# Wraping the vector store into a retriever object to fetch the most relevant documents for a given query using similarity search
retriever = vectorstore.as_retriever(
    search_type='similarity',                                                   # Use similarity search (based on vector distance)
    search_kwargs={'k': 3}                                                      # Retrieve top 2 most relevant documents
)

#### System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [ ]:
# Define the system prompt for the model
qna_system_message = """
You are an AI assistant designed to support professional doctors at St. Bernard's Medical Center. Your task is to provide evidence-based, concise, and relevant medical information to doctors' clinical questions based on the context provided.

User input will include the necessary context for you to answer their questions. This context will begin with the token: ###Context. The context contains references to specific portions of trusted medical literature and research articles relevant to the query, along with their source details.

When crafting your response:
1. Use only the provided context to answer the question.
2. If the answer is found in the context, respond with concise and actionable medical insights.
3. Include the source reference with the page number, journal name, or publication, as provided in the context.
4. If the question is unrelated to the context or the context is empty, clearly respond with: "Sorry, this is out of my knowledge base."

Please adhere to the following response guidelines:
- Provide clear, direct answers using only the given context.
- Do not include any additional information outside of the context.
- Avoid rephrasing or summarizing the context unless explicitly relevant to the question.
- If no relevant answer exists in the context, respond with: "Sorry, this is out of my knowledge base."
- If the context is not provided, your response should also be: "Sorry, this is out of my knowledge base."

Here is an example of how to structure your response:

Answer:
[Medical answer based on context]

Source:
[Source details with page or section]
"""

In [ ]:
# Define the user message template
qna_user_message_template = """
###Context
Here are some excerpts from medical literature and their sources that are relevant to the clinical question mentioned below:
{context}

###Question
{question}
"""

### Response Function

In [ ]:
def generate_rag_response(user_input,k=3,max_tokens=1000,temperature=0.75,top_p=0.95):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    # Generate the response
    try:
        response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": qna_system_message},
            {"role": "user", "content": user_message}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
        )
        # Extract and print the generated text from the response
        response = response.choices[0].message.content.strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Question1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
response_with_rag_1 = generate_rag_response(question_1)
response_with_rag_1

/tmp/ipython-input-2797984869.py:4: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)


'Answer:\nThe protocol for managing sepsis in a critical care unit includes the following steps:\n\n1. **Prompt Empiric Therapy**: Initiate very prompt empiric antibiotic therapy immediately after suspecting sepsis, after obtaining specimens for Gram stain and culture.\n   \n2. **Antibiotic Selection**: Choose antibiotics based on the suspected source, clinical setting, knowledge of likely causative organisms, and sensitivity patterns specific to the inpatient unit. Previous culture results should also be considered.\n\n3. **Example Regimen**: For septic shock of unknown cause, one regimen could include:\n   - Gentamicin or tobramycin 5.1 mg/kg IV once daily plus a 3rd-generation cephalosporin (e.g., cefotaxime 2 g every 6 to 8 hours or ceftriaxone 2 g once daily). If Pseudomonas is suspected, use ceftazidime 2 g IV every 8 hours.\n\n4. **Alternative Regimens**: Consider ceftazidime plus a fluoroquinolone (e.g., ciprofloxacin) or monotherapy with high doses of ceftazidime (2 g IV every

### Question2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
response_with_rag_2 = generate_rag_response(question_2)
response_with_rag_2

"Answer:\nThe common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia. After a few hours, the pain typically shifts to the right lower quadrant, and it increases with cough and motion. Classic signs include right lower quadrant direct and rebound tenderness at McBurney's point, Rovsing sign, and increased pain from passive extension of the right hip joint.\n\nAppendicitis cannot be cured via medicine alone; the treatment is surgical removal of the appendix, which can be performed through an open or laparoscopic appendectomy. \n\nSource:\nThe Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 11. Acute Abdomen & Surgical Gastroenterology, pages 163."

### Question3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
response_with_rag_3 = generate_rag_response(question_3)
response_with_rag_3

'Answer:\nEffective treatments for sudden patchy hair loss, known as alopecia areata, include topical, intralesional, or systemic corticosteroids in severe cases, topical minoxidil, topical anthralin, topical immunotherapy (e.g., diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). \n\nPossible causes behind alopecia areata include no obvious skin or systemic disorders, with the condition being characterized by sudden patchy hair loss.\n\nSource:\n[Source details not provided in the context.]'

### Question4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
response_with_rag_4 = generate_rag_response(question_4)
response_with_rag_4

'Answer:\nInitial treatment for a person who has sustained a traumatic brain injury (TBI) includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be necessary for severe injuries to monitor and treat intracranial pressure, decompress the brain, or remove hematomas. In the days following the injury, it is crucial to maintain adequate brain perfusion and oxygenation while preventing complications. Rehabilitation services should be initiated early, focusing on cognitive and emotional recovery, as well as addressing physical impairments.\n\nSource:\nThe Merck Manual of Diagnosis & Therapy, 19th Edition, Chapter 324. Traumatic Brain Injury.'

**Question 1 - Sepsis Management**

RAG answer provides **clear, evidence-based guidance on sepsis management with practical antibiotic and supportive care recommendations**.

**Question 2 - Appendicitis Symptoms and Treatment**

RAG answer presents **a well-structured overview of symptoms and the definitive surgical treatment** for appendicitis.

**Question 3 - Patchy Hair Loss (Alopecia Areata)**

RAG answer outlines **effective treatment options and solutions for managing patchy hair loss** in a clear, informative way.

**Question 4 - Brain Injury Treatment**

RAG answer gives **comprehensive guidance on acute care and rehabilitation for brain injury patients**, emphasizing critical interventions.


## Actionable Insights and Business Recommendations

1. **Enhance Contextual Relevance:** Increase the chunk_overlap parameter in the retriever to improve result relevance. Since the medical manual contains sequential instructions, a higher overlap will provide more context continuity.

2. **Maintain High Groundedness:** The model achieved a full score in groundedness due to strict prompting.

3. **Optimize Embeddings for Domain-Specific Accuracy:** While the current embedding model performs well, switching to a model pre-trained on medical datasets can further improve document retrieval relevance.

4. **Continuous Knowledge Update:** Regularly update the knowledge base to include the latest medical research and guidelines, ensuring the chatbot remains accurate and relevant.  

5. **Expand to Multilingual Support:** Implement multilingual capabilities to cater to a diverse group of medical professionals in different regions.  

6. **Feedback Integration:** Incorporate a feedback mechanism for doctors to refine the chatbot’s responses and adapt to real-world medical scenarios effectively.  

7. **Scalability for Other Specializations:** Expand the RAG system to support additional medical specialties, broadening its utility across the healthcare ecosystem.

<font size=6 color='blue'>Power Ahead</font>
___